# DiffuGPT-S: Issues 44 and 45 (T4 re-run)

This notebook re-runs **Issue 44 (diffusion refinement curves)** and **Issue 45 (found_time - unmask_time histograms)** on `diffusionfamily/diffugpt-s`, which is small enough (~0.1B params) for a free Colab **T4** GPU.

It is intentionally scoped to *only* issues 44 and 45 so the outputs are clean and easy to write up:

- **Issue 44** asks three things: (1) is unmasking deterministic / does a masked token unmask as the highest-probability final-layer token? (2) run the model on every history sequence and take the final-layer prediction, (3) plot the proportion of correct tokens over diffusion time, **stratified by task** (reasoning vs creative).
- **Issue 45** asks: for each token, find `found_time` (first diffusion step where the final-layer argmax equals the eventual token) and `unmask_time` (step the token is actually unmasked), histogram `delta = found_time - unmask_time` across many sequences, and print the large deltas with their text and token type.

Outputs are written to `diffugpt_s_44_45_outputs/` and a consolidated, paste-ready findings block is printed at the very end.

Runtime target: Colab/Jupyter with a T4 GPU.

## 1. Install dependencies and fetch DiffuGPT helper files

Run once at the top of a fresh runtime. If Colab asks you to restart after installs, restart and run this cell again.

In [ ]:
!pip -q install "torch" "transformers==4.44.2" "huggingface_hub" "safetensors" "pandas" "matplotlib" "tqdm" "accelerate"
!curl -L -o model.py https://raw.githubusercontent.com/HKUNLP/DiffuLLaMA/main/model.py
!curl -L -o attention_patch.py https://raw.githubusercontent.com/HKUNLP/DiffuLLaMA/main/attention_patch.py

## 2. Imports and configuration

Defaults are modest for a T4. `NUM_PROMPTS_PER_TASK` controls how many reasoning and creative sequences are generated; increase it for smoother histograms once a first pass works.

In [ ]:
import json
import math
import random
import string
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.distributions as dists
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer

from model import DiscreteDiffusionModel, get_anneal_attn_mask, top_p_logits

MODEL_NAME = "diffusionfamily/diffugpt-s"
BASE_MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Diffusion / generation settings.
DIFFUSION_STEPS = 64
GEN_LEN = 96
LOGITS_TEMP = 0.95
TOPP_TEMP = 0.9
SHIFT = True
SEED = 42

# How many prompts per task (reasoning / creative). 12 each -> 24 sequences is a
# good T4 starting point and gives enough tokens for the issue-45 histogram.
NUM_PROMPTS_PER_TASK = 12

# A token is a "large lead" (issue 45) if it was predicted this many steps before
# it was unmasked, i.e. lead = -delta = unmask_time - found_time >= threshold.
LARGE_LEAD_THRESHOLD = 8

OUT_DIR = Path("diffugpt_s_44_45_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Load DiffuGPT-S

In [ ]:
torch.manual_seed(SEED)
random.seed(SEED)

config = AutoConfig.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.mask_token_id is None:
    raise ValueError("DiffuGPT tokenizer is expected to define mask_token_id.")
MASK_ID = int(tokenizer.mask_token_id)

model = DiscreteDiffusionModel.from_pretrained(
    MODEL_NAME,
    model=BASE_MODEL_NAME,
    config=config,
    tokenizer=tokenizer,
    device=DEVICE,
).to(DEVICE)
model.eval()
if DEVICE == "cuda":
    model = model.half()

print("mask_token_id:", MASK_ID, repr(tokenizer.decode([MASK_ID])))
print("vocab_size:", model.vocab_size)

## 4. Build a stratified prompt bank

The `task` field (reasoning vs creative) is what we stratify the plots by. We expand a few templates with varied numbers/topics so the issue-45 histogram aggregates over many sequences.

In [ ]:
rng = random.Random(SEED)

REASONING_TEMPLATES = [
    "Question: A shop sold {a} items on Monday and half as many on Tuesday. How many in total? Answer step by step.\nAnswer:",
    "Question: A train travels {a} miles in {b} hours. What is its average speed in miles per hour? Explain briefly.\nAnswer:",
    "Question: A box has {a} red marbles and {b} blue marbles. What fraction are red? Explain step by step.\nAnswer:",
    "Question: What is {a} plus {b}? Show your reasoning step by step.\nAnswer:",
    "Question: If a book has {a} pages and you read {b} per day, how many days to finish? Explain.\nAnswer:",
    "Question: A class has {a} students and {b} are absent. How many are present? Answer step by step.\nAnswer:",
]

CREATIVE_TEMPLATES = [
    "Write a short, vivid paragraph about a city waking up after rain:\n",
    "Continue this story in a whimsical style: The old library only opened its hidden door when\n",
    "Describe a quiet morning in a {place} using vivid sensory detail:\n",
    "Write the opening of a story about a {object} that could remember the future:\n",
    "Describe the feeling of {place} at night in a single flowing paragraph:\n",
    "Tell a gentle tale about a {object} who wanted to see the sea:\n",
]
PLACES = ["a harbor town", "a mountain village", "an old forest", "a desert outpost", "a riverside market"]
OBJECTS = ["lantern", "music box", "compass", "paper boat", "clockwork bird"]

def build_prompts(num_per_task):
    prompts = []
    for i in range(num_per_task):
        tmpl = REASONING_TEMPLATES[i % len(REASONING_TEMPLATES)]
        prompts.append({
            "id": f"reasoning_{i}",
            "task": "reasoning",
            "prompt": tmpl.format(a=rng.randint(12, 96), b=rng.randint(2, 12)),
        })
    for i in range(num_per_task):
        tmpl = CREATIVE_TEMPLATES[i % len(CREATIVE_TEMPLATES)]
        prompts.append({
            "id": f"creative_{i}",
            "task": "creative",
            "prompt": tmpl.format(place=rng.choice(PLACES), object=rng.choice(OBJECTS)),
        })
    return prompts

PROMPTS = build_prompts(NUM_PROMPTS_PER_TASK)
print(f"{len(PROMPTS)} prompts ({NUM_PROMPTS_PER_TASK} per task)")
pd.DataFrame(PROMPTS).head()

## 5. Diffusion generation with histories

This mirrors the upstream DiffuGPT `generate_samples` loop, but at every diffusion step it stores:
- `xt_history`: the visible (masked/unmasked) sequence *before* the forward pass at that step
- `argmax_history`: the **final-layer argmax** token sequence for that step (shift-corrected)
- `final_ids`: the final generated sequence, used as ground truth

`argmax_history` is exactly the "run the model and take the final layer" quantity Issue 44 asks for, so we do not need a second forward pass.

In [ ]:
@dataclass
class HistoryResult:
    prompt_id: str
    task: str
    prompt: str
    prefix_len: int
    final_ids: list
    final_text: str
    xt_history: list
    argmax_history: list


def make_prefix_inputs(prompt, gen_len):
    prefix = [tokenizer.bos_token_id] + tokenizer.encode(prompt, add_special_tokens=False)
    if len(prefix) >= gen_len:
        prefix = prefix[: gen_len - 1]
    src_mask = [1] * len(prefix) + [0] * (gen_len - len(prefix))
    x0 = prefix + [0] * (gen_len - len(prefix))
    return {
        "input_ids": torch.tensor([x0], dtype=torch.long),
        "src_mask": torch.tensor([src_mask], dtype=torch.long),
        "prefix_len": len(prefix),
    }


def shifted_argmax_from_logits(logits, x, shift=True):
    raw = torch.argmax(logits, dim=-1)
    if shift:
        raw = torch.cat([x[:, 0:1], raw[:, :-1]], dim=1)
    return raw


@torch.inference_mode()
def generate_with_history(prompt_record, seed=SEED):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = make_prefix_inputs(prompt_record["prompt"], GEN_LEN)
    x = inputs["input_ids"].to(DEVICE)
    src_mask = inputs["src_mask"].bool().to(DEVICE)
    prefix_len = int(inputs["prefix_len"])

    x_embed = model.get_embeds(x)
    seq_len = x.size(1)
    batch_size = x.size(0)
    attention_mask = get_anneal_attn_mask(
        seq_len, batch_size, dtype=x_embed.dtype, device=x.device, attn_mask_ratio=1.0
    )

    maskable_mask = ~src_mask
    xt = x.masked_fill(maskable_mask, MASK_ID)

    xt_history = []
    argmax_history = []

    logits = model(xt, attention_mask=attention_mask)
    argmax_history.append(shifted_argmax_from_logits(logits, x, SHIFT).detach().cpu()[0].tolist())
    xt_history.append(xt.detach().cpu()[0].tolist())

    filter_logits = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
    scores = torch.log_softmax(filter_logits, dim=-1)
    x0 = dists.Categorical(logits=scores).sample()
    if SHIFT:
        x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
    x0 = xt.masked_scatter(maskable_mask, x0[maskable_mask])

    for t in range(DIFFUSION_STEPS - 1, 0, -1):
        p_to_x0 = 1 / (t + 1)
        masked_to_x0 = maskable_mask & (torch.rand_like(x0, dtype=torch.float) < p_to_x0)
        xt.masked_scatter_(masked_to_x0, x0[masked_to_x0])
        maskable_mask = maskable_mask.masked_fill(masked_to_x0, False)

        logits = model(xt, attention_mask=attention_mask)
        argmax_history.append(shifted_argmax_from_logits(logits, x, SHIFT).detach().cpu()[0].tolist())
        xt_history.append(xt.detach().cpu()[0].tolist())

        filter_logits = top_p_logits(logits / LOGITS_TEMP, p=TOPP_TEMP)
        scores = torch.log_softmax(filter_logits, dim=-1)
        x0 = dists.Categorical(logits=scores).sample()
        if SHIFT:
            x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
        x0 = xt.masked_scatter(maskable_mask, x0[maskable_mask])

    final_ids = x0.detach().cpu()[0].tolist()
    display_ids = final_ids[1:] if SHIFT else final_ids

    return HistoryResult(
        prompt_id=prompt_record["id"],
        task=prompt_record["task"],
        prompt=prompt_record["prompt"],
        prefix_len=prefix_len,
        final_ids=final_ids,
        final_text=tokenizer.decode(display_ids, skip_special_tokens=True),
        xt_history=xt_history,
        argmax_history=argmax_history,
    )

## 6. Run generation for all prompts

In [ ]:
histories = []
for i, record in enumerate(tqdm(PROMPTS, desc="generate histories")):
    hist = generate_with_history(record, seed=SEED + i)
    histories.append(hist)

print(f"generated {len(histories)} histories, {DIFFUSION_STEPS} steps each")
print("\nExample final text (", histories[0].prompt_id, "):\n", histories[0].final_text[:400])

## Issue 44, part 1: Is unmasking deterministic? Does a token unmask as the highest-probability final-layer token?

DiffuGPT does **not** unmask greedily. At each step it samples the revealed token from a top-p (p=`TOPP_TEMP`) filtered, temperature-scaled (`LOGITS_TEMP`) categorical distribution, and it randomly chooses *which* masked positions to reveal (`p_to_x0 = 1/(t+1)`). So the process is **stochastic**, not deterministic.

The cell below quantifies how often the token a position is eventually unmasked to matches the final-layer **argmax** at the step *just before* it was unmasked (i.e. while it was still masked). A value well below 1.0 confirms that unmasking is sampling-based rather than greedy argmax.

In [ ]:
def generated_positions(hist):
    return list(range(hist.prefix_len, GEN_LEN))

def first_unmask_time(hist, pos):
    for t, xt in enumerate(hist.xt_history):
        if int(xt[pos]) != MASK_ID:
            return t
    return None

match_argmax = 0
total_unmasked = 0
for hist in histories:
    for pos in generated_positions(hist):
        ut = first_unmask_time(hist, pos)
        if ut is None:
            continue
        total_unmasked += 1
        ref_t = max(ut - 1, 0)  # prediction while still masked (or step 0)
        final_tok = int(hist.final_ids[pos])
        if int(hist.argmax_history[ref_t][pos]) == final_tok:
            match_argmax += 1

frac = match_argmax / total_unmasked if total_unmasked else float("nan")
print(f"unmasked tokens analyzed: {total_unmasked}")
print(f"fraction unmasked-to-token == final-layer argmax (step before unmask): {frac:.4f}")
print("\nInterpretation: unmasking is stochastic top-p sampling, not greedy argmax;")
print("a fraction < 1.0 confirms the revealed token is not always the argmax token.")
determinism_summary = {"unmasked_tokens": total_unmasked, "frac_unmask_equals_argmax": frac}

## Issue 44, part 2-3: Diffusion refinement curves

For each diffusion step we count how many generated positions already have their final-layer argmax equal to the final (ground-truth) token, giving a `correct_fraction` per step. We also track `masked_fraction` (how many positions are still masked). Averaged within each task, this is the refinement curve.

In [ ]:
refinement_rows = []
for hist in histories:
    gpos = generated_positions(hist)
    gen_len = len(gpos)
    final = hist.final_ids
    n_steps = len(hist.xt_history)
    for t in range(n_steps):
        amax = hist.argmax_history[t]
        xt = hist.xt_history[t]
        correct = sum(1 for p in gpos if int(amax[p]) == int(final[p]))
        masked = sum(1 for p in gpos if int(xt[p]) == MASK_ID)
        refinement_rows.append({
            "prompt_id": hist.prompt_id,
            "task": hist.task,
            "step": t,
            "progress": t / max(n_steps - 1, 1),
            "num_generated_tokens": gen_len,
            "correct_fraction": correct / gen_len if gen_len else 0.0,
            "masked_fraction": masked / gen_len if gen_len else 0.0,
        })

refinement_df = pd.DataFrame(refinement_rows)
refinement_df.to_csv(OUT_DIR / "refinement_by_step.csv", index=False)
print(refinement_df.groupby("task")["correct_fraction"].describe()[["mean", "min", "max"]])

### Plot Issue 44: refinement over diffusion progress, stratified by task

The question is whether refinement is steady/linear or concentrated in certain parts of the diffusion process, and whether reasoning and creative tasks differ.

In [ ]:
curve = refinement_df.groupby(["task", "step"], as_index=False).agg(
    progress=("progress", "mean"),
    correct_fraction=("correct_fraction", "mean"),
    masked_fraction=("masked_fraction", "mean"),
)

plt.figure(figsize=(9, 5))
for task, sub in curve.groupby("task"):
    sub = sub.sort_values("progress")
    plt.plot(sub["progress"], sub["correct_fraction"], linewidth=2.4, label=f"{task}: correct argmax")
    plt.plot(sub["progress"], 1 - sub["masked_fraction"], linewidth=1.4, linestyle="--",
             alpha=0.7, label=f"{task}: unmasked fraction")
plt.xlabel("Diffusion progress")
plt.ylabel("Fraction")
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend(fontsize=8)
plt.title("Issue 44: DiffuGPT-S refinement over diffusion time")
plt.tight_layout()
plt.savefig(OUT_DIR / "issue44_refinement_curve.png", dpi=200)
plt.show()

# How concentrated is refinement? Report progress to reach 50% / 90% of final correctness.
issue44_summary = {}
for task, sub in curve.groupby("task"):
    sub = sub.sort_values("progress")
    final_corr = sub["correct_fraction"].iloc[-1]
    def progress_to(frac_of_final):
        thr = frac_of_final * final_corr
        hit = sub[sub["correct_fraction"] >= thr]
        return float(hit["progress"].iloc[0]) if len(hit) else float("nan")
    auc = float((sub["correct_fraction"]).mean())  # mean height ~ area under curve
    issue44_summary[task] = {
        "final_correct_fraction": float(final_corr),
        "progress_to_50pct": progress_to(0.5),
        "progress_to_90pct": progress_to(0.9),
        "mean_correct_fraction_auc": auc,
    }
print(json.dumps(issue44_summary, indent=2))

## Issue 45: found_time, unmask_time, and the delta histogram

For each generated token (ground truth = final sequence):
- `found_time` = first diffusion step where the final-layer argmax equals the final token.
- `unmask_time` = first step where the position is actually unmasked.
- `delta = found_time - unmask_time`. **Negative** delta means the model "knew" the token before it was unmasked; `lead = -delta` is how many steps early.

In [ ]:
STOPWORDS = set("the a an and or but if then of to in on at for with as is are was were be been by this that these those it its he she they we you i".split())

def classify_token(token_id):
    text = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    s = text.strip()
    if s == "":
        return "whitespace"
    if all(ch in string.punctuation for ch in s):
        return "punctuation"
    if any(ch.isdigit() for ch in s):
        return "number"
    if s.lower() in STOPWORDS:
        return "function_word"
    return "content_word"

def first_found_time(hist, pos, final_tok):
    for t, amax in enumerate(hist.argmax_history):
        if int(amax[pos]) == int(final_tok):
            return t
    return None

def first_unmask_as_final(hist, pos, final_tok):
    for t, xt in enumerate(hist.xt_history):
        tok = int(xt[pos])
        if tok != MASK_ID and tok == int(final_tok):
            return t
    return None

token_delta_rows = []
for hist in histories:
    for pos in generated_positions(hist):
        final_tok = int(hist.final_ids[pos])
        found = first_found_time(hist, pos, final_tok)
        unmask = first_unmask_as_final(hist, pos, final_tok)
        delta = None if (found is None or unmask is None) else (found - unmask)
        token_delta_rows.append({
            "prompt_id": hist.prompt_id,
            "task": hist.task,
            "pos": pos,
            "final_token_id": final_tok,
            "final_token_str": tokenizer.decode([final_tok], skip_special_tokens=False),
            "token_class": classify_token(final_tok),
            "found_time": found,
            "unmask_time": unmask,
            "delta_found_minus_unmask": delta,
            "lead_steps": None if delta is None else -delta,
        })

token_delta_df = pd.DataFrame(token_delta_rows)
token_delta_df.to_csv(OUT_DIR / "token_deltas.csv", index=False)
valid = token_delta_df.dropna(subset=["delta_found_minus_unmask"])
print(f"tokens with valid delta: {len(valid)} / {len(token_delta_df)}")
print(valid.groupby("task")["delta_found_minus_unmask"].describe()[["count", "mean", "min", "max"]])

### Plot Issue 45: histogram of `found_time - unmask_time`, stratified by task

In [ ]:
work = valid.copy()
work["delta_found_minus_unmask"] = work["delta_found_minus_unmask"].astype(int)

plt.figure(figsize=(9, 5))
if len(work):
    lo = int(work["delta_found_minus_unmask"].min())
    hi = int(work["delta_found_minus_unmask"].max())
    bins = range(lo, hi + 2)
    for task, sub in work.groupby("task"):
        plt.hist(sub["delta_found_minus_unmask"], bins=bins, alpha=0.5, label=task)
plt.axvline(0, color="black", linewidth=1.2)
plt.xlabel("found_time - unmask_time  (negative = predicted before unmasking)")
plt.ylabel("token count")
plt.legend()
plt.grid(alpha=0.2)
plt.title("Issue 45: time between prediction and unmasking")
plt.tight_layout()
plt.savefig(OUT_DIR / "issue45_delta_histogram.png", dpi=200)
plt.show()

issue45_summary = {
    "num_valid_tokens": int(len(work)),
    "delta_mean": float(work["delta_found_minus_unmask"].mean()),
    "delta_median": float(work["delta_found_minus_unmask"].median()),
    "frac_predicted_before_unmask": float((work["delta_found_minus_unmask"] < 0).mean()),
    "frac_at_unmask": float((work["delta_found_minus_unmask"] == 0).mean()),
    "max_lead_steps": int((-work["delta_found_minus_unmask"]).max()),
}
print(json.dumps(issue45_summary, indent=2))

### Issue 45: large leads — tokens predicted long before unmasking

These are tokens with `lead_steps >= LARGE_LEAD_THRESHOLD`. We print the token, how many steps early it was predicted, its token class, and the sentence it came from, then break down what *kinds* of tokens dominate the large-lead set.

In [ ]:
leads = work.copy()
leads["lead_steps"] = -leads["delta_found_minus_unmask"]
large = leads[leads["lead_steps"] >= LARGE_LEAD_THRESHOLD].sort_values("lead_steps", ascending=False)

final_text_by_id = {h.prompt_id: h.final_text for h in histories}
large_records = []
print(f"{len(large)} large-lead tokens (lead >= {LARGE_LEAD_THRESHOLD}). Top 25:\n")
for _, r in large.head(25).iterrows():
    rec = {
        "prompt_id": r["prompt_id"],
        "task": r["task"],
        "token": r["final_token_str"],
        "token_class": r["token_class"],
        "lead_steps": int(r["lead_steps"]),
        "found_time": int(r["found_time"]),
        "unmask_time": int(r["unmask_time"]),
        "sentence": final_text_by_id.get(r["prompt_id"], "")[:200],
    }
    large_records.append(rec)
    print(f"[{rec['task']}] lead={rec['lead_steps']:>3}  {rec['token']!r:>14}  ({rec['token_class']})  in: {rec['sentence']!r}")

with open(OUT_DIR / "large_leads.json", "w") as f:
    json.dump(large_records, f, indent=2, ensure_ascii=False)

print("\nLarge-lead token-class breakdown:")
if len(large):
    print(large["token_class"].value_counts())
    print("\nAll-token class breakdown (for comparison):")
    print(leads["token_class"].value_counts())

## Save outputs and print a paste-ready findings block

The block below consolidates the key numbers for both issues so they can be dropped straight into the writeup.

In [ ]:
summaries = {
    "config": {
        "model": MODEL_NAME, "diffusion_steps": DIFFUSION_STEPS, "gen_len": GEN_LEN,
        "logits_temp": LOGITS_TEMP, "topp_temp": TOPP_TEMP, "seed": SEED,
        "num_prompts": len(PROMPTS), "num_prompts_per_task": NUM_PROMPTS_PER_TASK,
        "large_lead_threshold": LARGE_LEAD_THRESHOLD,
    },
    "issue44_determinism": determinism_summary,
    "issue44_refinement": issue44_summary,
    "issue45": issue45_summary,
}
with open(OUT_DIR / "summaries.json", "w") as f:
    json.dump(summaries, f, indent=2)

print("==================== WRITEUP-READY FINDINGS ====================")
print(json.dumps(summaries, indent=2))
print("\nFiles written to", OUT_DIR.resolve(), ":")
for p in sorted(OUT_DIR.iterdir()):
    print("  ", p.name)

In [ ]:
# Colab-only convenience download. In local Jupyter, use the file browser instead.
import shutil
shutil.make_archive("diffugpt_s_44_45_outputs", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("diffugpt_s_44_45_outputs.zip")
except Exception as exc:
    print("Download helper skipped:", exc)